In [0]:
df = spark.table("bronze.customer_complaints")
df.createOrReplaceTempView("bronze_raw")

In [0]:
total_bronze_rows = df.count()
print(f"Total rows read from Bronze layer: {total_bronze_rows}")

In [0]:
# when new complaints were added to the bronze layer  
display(df.groupBy("ingestion_timestamp").count())

In [0]:
# check for blank IDs (if any, the quarantine table will be 0)
from pyspark.sql import functions as F
display(df.filter(F.col("Complaint_ID").isNull()))

In [0]:
# isolating clean raws and dirty raws 
from pyspark.sql import functions as F 

raw_df = spark.table("bronze_raw")

quarantine_df = raw_df.filter((F.col("Complaint_ID").isNull()) | (F.trim(F.col("Complaint_ID")) == ""))
quarantine_rows_count = quarantine_df.count()

clean_df = raw_df.filter((F.col("Complaint_ID").isNotNull()) & (F.trim(F.col("Complaint_ID")) != ""))

In [0]:
# in case there are no ID nulls, the quarintine table will be empty so we handle it by only creating
# the quarntine table if any ID nulls exist  
if quarantine_rows_count > 0:
    quarantine_df.write.format("delta").mode("append").saveAsTable("silver.complaints_quarantine")
    print(f"{quarantine_rows_count} bad rows written to quarantine table.")
else:
    print("No bad rows found in this batch. Quarantine table skip completed safely.")

In [0]:
# drop the empty column 
clean_df = clean_df.drop("Consumer_disputed")

# trim all the collumns 
string_columns = [item[0] for item in clean_df.dtypes if item[1] == "string"]
for col_name in string_columns :
    clean_df = clean_df.withColumn(col_name, F.trim(F.col(col_name)))


In [0]:
# change the column's name to snake_case format 
def to_snake_case(name) :
    return name.replace(" ", "_").replace("?","").lower()

for old_col in clean_df.columns:
    clean_df = clean_df.withColumnRenamed(old_col, to_snake_case(old_col))


In [0]:
clean_df.createOrReplaceTempView("silver_step1")

clean_rows_count = clean_df.count()

# this formula ensures that there are no missing rows 
assert total_bronze_rows == (clean_rows_count + quarantine_rows_count)

print(f"Step 1 Success!")
print(f"Total Bronze: {total_bronze_rows}")      
print(f"Silver Clean View: {clean_rows_count}")   
print(f"Quarantined Rows: {quarantine_rows_count}")